In [1]:
import pandas as pd
from dotenv import load_dotenv
import os

from langchain_community.graphs import Neo4jGraph

load_dotenv()  # loads variables from .env

NEO4J_URL = os.getenv("NEO4J_URL")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

data = pd.read_csv("../data/oil_imports_23_24_cln.csv")

data = data[['period', 'reporterDesc', 'partnerDesc', 'final_netWgt_MT','primaryValue',
       'final_price_per_mt', 'barrels' ]].drop_duplicates()
print(data.shape)

(8284, 7)


In [7]:
graph = Neo4jGraph(
    url=NEO4J_URL,
    username=NEO4J_USER,
    password=NEO4J_PASSWORD,
    database="78ce8520",
    refresh_schema=True 
)

# 1. Ensure Uniqueness and Indexing
graph.query("CREATE CONSTRAINT country_name_unique IF NOT EXISTS FOR (c:Country) REQUIRE c.name IS UNIQUE")
graph.query("CREATE CONSTRAINT year_month_id_unique IF NOT EXISTS FOR (t:YearMonth) REQUIRE t.yearMonth IS UNIQUE")

[]

In [3]:
print(graph.schema)

Node properties:

Relationship properties:

The relationships:



In [5]:
# Cypher query to load your preprocessed data
ingest_query = """
UNWIND $data as row
// 1. Ensure all nodes exist uniquely
MERGE (r:Country {name: row.reporterDesc})
MERGE (p:Country {name: row.partnerDesc})
MERGE (t:YearMonth {yearMonth: row.period})

// 2. MERGE the trade relationship (The primary data link)
// We include the period in the pattern to prevent 63.8B duplicate error
MERGE (r)-[f:IMPORTED_FROM {year_month: row.period}]->(p)

// 3. Connect the Importer to the Time node (The correct way to link time)
MERGE (r)-[:IMPORTS_IN]->(t)

// 4. Set the numerical data
SET f.barrels = row.barrels,
    f.value_usd = row.primaryValue,
    f.price_mt = row.final_price_per_mt
"""

# Prepare your data from the cleaned dataframe
data_to_load = data.to_dict('records')

# Execute ingestion
graph.query(ingest_query, params={"data": data_to_load})

[]

In [8]:
print(graph.schema)

Node properties:
Country {name: STRING}
YearMonth {yearMonth: INTEGER}
Relationship properties:
IMPORTED_FROM {barrels: FLOAT, value_usd: FLOAT, price_mt: FLOAT, year_month: INTEGER}
The relationships:
(:Country)-[:IMPORTS_IN]->(:YearMonth)
(:Country)-[:IMPORTED_FROM]->(:Country)


```cypher
\\ DELETE NODES AND RELATIONSHIPS
MATCH (n)
DETACH DELETE n

\\ DELETE ONLY RELATIONSHIPS
MATCH ()-[r:IMPORTED_FROM]->()
DELETE r

\\ DROP CONSTRAINTS
DROP CONSTRAINT country_name_unique IF EXISTS;
DROP CONSTRAINT period_id_unique IF EXISTS;